# Using the IBM dataset with Kosh and Pytorch


Assumes you are on LC machine.

In [1]:
import os
import kosh

# Make sure local file is new sql file
kosh_example_sql_file = "/p/lscratchh/cdoutrix/cdoutrix/IBM/workaround/kosh_example.sql"

In [2]:
from  kosh import KoshStore
import os

# connect to store
store = KoshStore(engine="sina", username=os.environ["USER"], sql='sql', db_path=kosh_example_sql_file)

In [3]:
from sina.utils import DataRange
train_set = store.search(project="IBM", taperThresh=DataRange(.6), skewThresh=DataRange(.6), ids_only=True)
print(len(train_set))
test_set = list(set(store.search(project="IBM", ids_only=True)).symmetric_difference(train_set))

144


In [4]:
import numpy
from torch.utils.data import Dataset
import aml_dmt

class IBMDataset(Dataset):
    def __init__(self, kosh_datasets, label_metric, features_metrics, symmetry=2, history_length=100):
        Dataset.__init__(self)
        # open and store datasets from kosh
        self.datasets = [store.open(ds) for ds in kosh_datasets]
        self.label_metric = label_metric
        self.datasets_length = []
        self.symmetry = symmetry
        self.features_metrics = features_metrics
        self.history_length = history_length
        for ds in self.datasets:
            self.datasets_length += [self.dataset_length(ds),]

    def dataset_length(self, ds):
        n = 0
        bad = ds.bad_nodes[self.label_metric]
        for node in bad:
            n += len(aml_dmt.ibm.string2indices(bad[node]))
        return n * self.symmetry
        
    def __len__(self):
        n = 0
        for length in self.datasets_length:
            n += length
        return n
        
    def __getitem__(self, key):
        # figure out which dataset to use
        total = 0
        for ds_index, length in enumerate(self.datasets_length):
            total += length
            if total > key:
                break
        # offset 0 means first node in this dataset
        offset = total - length
        # index in dataset
        node_index = (key - offset) // self.symmetry
        # bad node type or good node type
        node_type = key % self.symmetry
        # sort the nodes so it is always the same node for each index
        node_keys = sorted([int(k) for k in self.datasets[ds_index].bad_nodes[self.label_metric]])
        index_in_bads = 0
        bads = self.datasets[ds_index].bad_nodes[self.label_metric]
        for node in bads:
            indices = aml_dmt.ibm.string2indices(bads[node])
            n = len(indices)
            if index_in_bads + n < node_index:
                index_in_bads += n
            else:
                time_index = indices[index_in_bads + n - node_index - 1]
        node = int(node)
        #time_index = index_in_bads
        h5 = self.datasets[ds_index].search(mime_type="hdf5")[0].open()
        ntimes = h5["node/metrics_0"].shape[0]
        if node_type == 0:  # we want a bad node
            # nothing to do we're good
            pass
        else:
            # skip a few nodes
            node += node_type
            if node > ntimes: # make sure it is the valid range
                node -= ntimes
            while str(node) in bads:
                # skip a few nodes
                node += self.symmetry
                if node > ntimes: # make sure it is the valid range
                    node -= ntimes
        out = None
        for metric in self.features_metrics:
            #print(metric, time_index, node)
            data = h5["node/{}".format(metric)][time_index-self.history_length+1:time_index+1, node]
            #data = data.reshape((data.shape[0],1))
            # We probably should normalize here...
            if out is None:
                out = data
            else:
                out = numpy.hstack((out, data))
        # Numpy to torch?
        return out,numpy.array(node_type)

In [10]:
features = ["metrics_{}".format(i) for i in [20,23]]
history = 20

ts = sorted(train_set[:4])
ibm = IBMDataset(ts, "metrics_3", features, history_length=history)
print(len(ibm))
for i in range(len(ibm)):
    print(ibm[i][0].shape)

642
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)
(40,)


In [11]:
from torch.utils.data import DataLoader

train_loader = DataLoader(ibm, batch_size=4,
                        shuffle=False, num_workers=4)



In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from tqdm.autonotebook import tqdm

# class SqueezeDim(nn.Module):
#     def __init__(self, dim):
#         super(SqueezeDim, self).__init__()
#         self.dim = dim
#     def forward(self, x):
#         return x.squeeze(self.dim)

length = history*len(features)
model = nn.Sequential(
    nn.Linear(length,10),
    nn.ReLU(),
    nn.Linear(10,10),
    nn.ReLU(),
    nn.Linear(10,10),
    nn.ReLU(),
    nn.Linear(10,2),
    #SqueezeDim(-1),
  )

loss = nn.CrossEntropyLoss()

def train(model, train_loader, optimizer, epoch):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        #data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        ll = loss(output,target)
        ll.backward()
        optimizer.step()
        if 1:
            print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, batch_idx * len(data), len(train_loader.dataset),
                100. * batch_idx / len(train_loader), ll.item()))

optimizer = optim.SGD(model.parameters(), lr=1e-2, momentum=0.1)

train(model, train_loader, optimizer, 0)




Train Epoch: 0 [0/642 (0%)]	Loss: 0.696634
Train Epoch: 0 [4/642 (1%)]	Loss: 0.695207
Train Epoch: 0 [8/642 (1%)]	Loss: 0.696631
Train Epoch: 0 [12/642 (2%)]	Loss: 0.698740
Train Epoch: 0 [16/642 (2%)]	Loss: 0.698576
Train Epoch: 0 [20/642 (3%)]	Loss: 0.697323
Train Epoch: 0 [24/642 (4%)]	Loss: 0.697214
Train Epoch: 0 [28/642 (4%)]	Loss: 0.698313
Train Epoch: 0 [32/642 (5%)]	Loss: 0.697152
Train Epoch: 0 [36/642 (6%)]	Loss: 0.697171
Train Epoch: 0 [40/642 (6%)]	Loss: 0.696043
Train Epoch: 0 [44/642 (7%)]	Loss: 0.695793
Train Epoch: 0 [48/642 (7%)]	Loss: 0.695774
Train Epoch: 0 [52/642 (8%)]	Loss: 0.696015
Train Epoch: 0 [56/642 (9%)]	Loss: 0.697928
Train Epoch: 0 [60/642 (9%)]	Loss: 0.696886


TypeError: __init__() missing 2 required positional arguments: 'params' and 'orig'